# 01: Quantitative Research, Strategy Validation & Backtesting
## ETHUSDT Perpetual Futures Research Environment

This is the primary research notebook for:
- Inspecting canonical market data.
- Computing causal features and technical indicators.
- Running event-aware backtests with realistic transaction costs & slippage.
- Analyzing comprehensive statistical performance metrics (Sharpe, Sortino, R-multiples, MFE/MAE).
- Permanently recording results and negative outcomes into the **Research Ledger**.

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().resolve()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import polars as pl
from quant_platform.data.storage.canonical import CanonicalStorage
from quant_platform.data.timeframes.resampler import CausalResampler
from quant_platform.features import compute_ema, compute_rsi, compute_atr, CausalSwingEngine
from quant_platform.strategies import EmaTrendStrategy, RsiMeanReversionStrategy, BreakoutSanityStrategy
from quant_platform.backtest import BacktestEngine, CostModel
from quant_platform.research import ResearchLedger, MLflowAdapter, ReportGenerator
from quant_platform.domain.experiment import ExperimentRecord, ExperimentStatus

print("Research environment initialized!")

### 1. Load Canonical Data & Resample to 15-Minute Causal Timeframe

In [ ]:
storage = CanonicalStorage()
df_1m = storage.read_range(symbol="ETHUSDT", timeframe="1m")

if df_1m.is_empty():
    print("Canonical data not found on disk. Please fetch data first via Notebook 00 or `quant data fetch`.")
else:
    # Strictly causal resampling to 15m
    df_15m = CausalResampler.resample(df_1m, target_timeframe="15m")
    print(f"Loaded {len(df_15m)} 15-minute bars.")
    print(df_15m.head(5))

### 2. Compute Causal Indicators & Market Structure Swings

In [ ]:
if not df_1m.is_empty():
    df_features = compute_ema(df_15m, period=20, output_col="ema_20")
    df_features = compute_ema(df_features, period=50, output_col="ema_50")
    df_features = compute_rsi(df_features, period=14, output_col="rsi_14")
    df_features = compute_atr(df_features, period=14, output_col="atr_14")
    df_features = CausalSwingEngine.detect_swings(df_features, left_bars=5, right_bars=5)
    print("Features computed successfully!")
    print(df_features.select(["open_time", "close", "ema_20", "ema_50", "rsi_14", "atr_14", "last_swing_high_L5_R5"]).tail(5))

### 3. Configure Strategy & Execute Event-Aware Backtest

In [ ]:
if not df_1m.is_empty():
    strategy = EmaTrendStrategy(fast_period=20, slow_period=50, atr_multiplier_stop=2.0, risk_reward_ratio=2.0)
    engine = BacktestEngine(
        cost_model=CostModel(maker_fee_rate=0.0002, taker_fee_rate=0.0005, slippage_bps=2.0),
        initial_capital=10000.0,
        risk_per_trade_fraction=0.01,
    )

    result = engine.run(df_15m, strategy)
    m = result.metrics

    print("=== BACKTEST PERFORMANCE METRICS ===")
    print(f"Total Net Return:   {m.total_net_return:+.2f}%")
    print(f"Win Rate:           {m.win_rate:.1f}% ({m.trade_count} trades)")
    print(f"Profit Factor:      {m.profit_factor:.2f}")
    print(f"Sharpe Ratio:       {m.sharpe_ratio:.2f}")
    print(f"Sortino Ratio:      {m.sortino_ratio:.2f}")
    print(f"Max Drawdown:       {m.max_drawdown_pct:.2f}%")
    print(f"Average R:          {m.average_r:.2f}R")
    print(f"Total Fees Paid:    ${m.total_fees:.2f}")
    print(f"Intrabar Ambiguities: {m.intrabar_ambiguity_count}")

### 4. Register Experiment in Persistent Research Ledger & Generate Report

In [ ]:
if not df_1m.is_empty():
    ledger = ResearchLedger()
    exp_id = ledger.generate_experiment_id()

    record = ExperimentRecord(
        experiment_id=exp_id,
        hypothesis=strategy.metadata.hypothesis,
        strategy_id=strategy.metadata.strategy_id,
        strategy_version=strategy.metadata.version,
        feature_set_version="v1",
        parameters=strategy.metadata.parameters,
        parameter_hash=ledger._compute_parameter_hash(strategy.metadata.parameters),
        dataset_fingerprint="canonical_ethusdt_15m",
        symbol="ETHUSDT",
        timeframe="15m",
        date_range_start="2024-01-01",
        date_range_end="2024-01-31",
        cost_model=engine.cost_model.model_dump(),
        execution_model="EVENT_AWARE_TAKER_ENTRY_LIMIT_TP_MARKET_SL",
        risk_model="STRUCTURAL_ATR_FIXED_RISK_1PCT",
        metrics=m,
        trade_count=m.trade_count,
        status=ExperimentStatus.COMPLETED if m.total_net_return > 0 else ExperimentStatus.REJECTED,
        conclusion="Baseline EMA trend validation completed.",
    )

    # Register experiment
    ledger.register_experiment(record)

    # Generate HTML & JSON reports
    reporter = ReportGenerator()
    html_path = reporter.generate_html_report(record, result.ledger)
    json_path = reporter.generate_json_export(record)
    print(f"Experiment registered: {exp_id}")
    print(f"HTML Report: {html_path}")